In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.metrics import mean_squared_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
try:
    from google.colab import files
    print("Загрузите файл train.csv")
    uploaded = files.upload()

    file_name = list(uploaded.keys())[0]

    df_raw = pd.read_csv(file_name)
    print("Размеры датафрейма:", df_raw.shape)
    display(df_raw.head())

except Exception as e:
    print("Ошибка при загрузке:", e)

Загрузите файл train.csv


Saving train.csv to train (3).csv
Размеры датафрейма: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [ ]:
NUM_FEATURES = [
    'OverallQual','GrLivArea','GarageCars','GarageArea',
    'TotalBsmtSF','1stFlrSF','FullBath','YearBuilt',
    'YearRemodAdd','TotRmsAbvGrd','Fireplaces','LotArea',
    'MasVnrArea','GarageYrBlt'
]

TARGET = 'SalePrice'
K = 7

df = df_raw[NUM_FEATURES + [TARGET]].dropna()

print("Размер после удаления пропусков:", df.shape)
df.head()

Размер после удаления пропусков: (1371, 15)


,OverallQual,GrLivArea,GarageCars,GarageArea,TotalBsmtSF,1stFlrSF,FullBath,YearBuilt,YearRemodAdd,TotRmsAbvGrd,Fireplaces,LotArea,MasVnrArea,GarageYrBlt,SalePrice
0,7,1710,2,548,856,856,2,2003,2003,8,0,8450,196.0,2003.0,208500
1,6,1262,2,460,1262,1262,2,1976,1976,6,1,9600,0.0,1976.0,181500
2,7,1786,2,608,920,920,2,2001,2002,6,1,11250,162.0,2001.0,223500
3,7,1717,3,642,756,961,1,1915,1970,7,1,9550,0.0,1998.0,140000
4,8,2198,3,836,1145,1145,2,2000,2000,9,1,14260,350.0,2000.0,250000


In [ ]:
X = df[NUM_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("Форма X_train:", X_train.shape)
print("Форма X_test :", X_test.shape)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

Форма X_train: (1096, 14)
Форма X_test : (275, 14)


In [ ]:
lin_all = LinearRegression()
lin_all.fit(X_train_scaled, y_train)

y_pred_all = lin_all.predict(X_test_scaled)

mse_all = mean_squared_error(y_test, y_pred_all)
r2_all = r2_score(y_test, y_pred_all)

print(f"Baseline (все 14 признаков) -> MSE: {mse_all:.2f}, R2: {r2_all:.4f}")

Baseline (все 14 признаков) -> MSE: 1210835854.27, R2: 0.8139


In [ ]:
#SelectKBest (f_regression, k=7)
selector = SelectKBest(score_func=f_regression, k=K)
X_train_kbest = selector.fit_transform(X_train_scaled, y_train)
X_test_kbest  = selector.transform(X_test_scaled)

selected_mask = selector.get_support()
selected_features = np.array(NUM_FEATURES)[selected_mask]

print("Отобранные признаки SelectKBest:")
print(selected_features)

lin_kbest = LinearRegression()
lin_kbest.fit(X_train_kbest, y_train)

y_pred_kbest = lin_kbest.predict(X_test_kbest)

mse_kbest = mean_squared_error(y_test, y_pred_kbest)
r2_kbest = r2_score(y_test, y_pred_kbest)

print(f"\nSelectKBest -> MSE: {mse_kbest:.2f}, R2: {r2_kbest:.4f}")

Отобранные признаки SelectKBest:
['OverallQual' 'GrLivArea' 'GarageCars' 'GarageArea' 'TotalBsmtSF'
 '1stFlrSF' 'FullBath']

SelectKBest -> MSE: 1346143793.06, R2: 0.7931


С какой проблемой вы столкнётесь, если будете оценивать 50 признаков корреляцией Пирсона?
При использовании только корреляции Пирсона каждый признак оценивается отдельно, без учёта взаимодействий между признаками. Поэтому, если у нас 50 признаков, некоторые из них могут быть полезны только в комбинациях, но иметь слабую индивидуальную корреляцию с целевой переменной. В итоге фильтрующий метод может отбросить важные признаки или выбрать не самые подходящие.

In [ ]:
#RFE (рекурсивное исключение признаков, k=7)
rfe = RFE(estimator=LinearRegression(), n_features_to_select=K)
rfe.fit(X_train_scaled, y_train)

rfe_mask = rfe.get_support()
rfe_features = np.array(NUM_FEATURES)[rfe_mask]

print("Отобранные признаки RFE:")
print(rfe_features)

X_train_rfe = rfe.transform(X_train_scaled)
X_test_rfe  = rfe.transform(X_test_scaled)

lin_rfe = LinearRegression()
lin_rfe.fit(X_train_rfe, y_train)

y_pred_rfe = lin_rfe.predict(X_test_rfe)

mse_rfe = mean_squared_error(y_test, y_pred_rfe)
r2_rfe = r2_score(y_test, y_pred_rfe)

print(f"\nRFE -> MSE: {mse_rfe:.2f}, R2: {r2_rfe:.4f}")


Отобранные признаки RFE:
['OverallQual' 'GrLivArea' 'GarageCars' 'TotalBsmtSF' 'YearRemodAdd'
 'Fireplaces' 'MasVnrArea']

RFE -> MSE: 1273001526.47, R2: 0.8044


RFE считает качество всей модели в целом и постепенно исключает самые слабые признаки. Благодаря этому он может находить лучшие комбинации признаков, даже если отдельные признаки сами по себе слабые. В отличие от фильтров, RFE учитывает взаимодействия между признаками и их вклад в итоговую модель.

In [ ]:
#RandomForest Feature Importance
rf = RandomForestRegressor(
    n_estimators=300,
    random_state=RANDOM_STATE
)

rf.fit(X_train, y_train)

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

print("Важности признаков RandomForest:")
for i in range(len(NUM_FEATURES)):
    print(f"{NUM_FEATURES[indices[i]]}: {importances[indices[i]]:.4f}")

#Берём 7 лучших
top_rf_features = [NUM_FEATURES[i] for i in indices[:K]]
print("\n7 лучших признаков по RandomForest:")
print(top_rf_features)

X_train_rf = X_train[top_rf_features]
X_test_rf  = X_test[top_rf_features]

#Масштабируем
scaler_rf = StandardScaler()
X_train_rf_scaled = scaler_rf.fit_transform(X_train_rf)
X_test_rf_scaled  = scaler_rf.transform(X_test_rf)

lin_rf = LinearRegression()
lin_rf.fit(X_train_rf_scaled, y_train)

y_pred_rf = lin_rf.predict(X_test_rf_scaled)

mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"\nRandomForest importance -> MSE: {mse_rf:.2f}, R2: {r2_rf:.4f}")

Важности признаков RandomForest:
OverallQual: 0.5795
GrLivArea: 0.1406
TotalBsmtSF: 0.0528
1stFlrSF: 0.0401
LotArea: 0.0276
GarageArea: 0.0272
GarageCars: 0.0228
YearBuilt: 0.0226
YearRemodAdd: 0.0203
MasVnrArea: 0.0197
FullBath: 0.0130
TotRmsAbvGrd: 0.0128
GarageYrBlt: 0.0118
Fireplaces: 0.0092

7 лучших признаков по RandomForest:
['OverallQual', 'GrLivArea', 'TotalBsmtSF', '1stFlrSF', 'LotArea', 'GarageArea', 'GarageCars']

RandomForest importance -> MSE: 1264379776.58, R2: 0.8057


Почему L1 выполняет отбор признаков, а L2 — нет?

L1-регуляризация (Lasso) может обнулять коэффициенты признаков, полностью исключая их из модели. Поэтому она работает как встроенный метод отбора признаков.
L2-регуляризация (Ridge) делает коэффициенты меньше, но никогда не обнуляет их. Поэтому она не удаляет признаки, а просто уменьшает вклад каждого из них.



In [ ]:
results = pd.DataFrame({
    'Method': [
        'Baseline (14 признаков)',
        'SelectKBest (7 признаков)',
        'RFE (7 признаков)',
        'RandomForest Importance (7 признаков)'
    ],
    'MSE': [
        mse_all,
        mse_kbest,
        mse_rfe,
        mse_rf
    ],
    'R2': [
        r2_all,
        r2_kbest,
        r2_rfe,
        r2_rf
    ]
})

results

,Method,MSE,R2
0,Baseline (14 признаков),1.210836e+09,0.813933
1,SelectKBest (7 признаков),1.346144e+09,0.793140
2,RFE (7 признаков),1.273002e+09,0.804380
3,RandomForest Importance (7 признаков),1.264380e+09,0.805705


В ходе лабораторной работы я сравнил несколько методов отбора признаков для задачи предсказания стоимости домов. Сначала была обучена базовая модель линейной регрессии на всех 14 числовых признаках. Она показала наилучший результат с R² = 0.8139, что логично, так как в ней используется максимальное количество информации.

Далее я применил три метода отбора признаков: SelectKBest, RFE и RandomForest Importance. Каждый из них оставлял по 7 признаков. По итогам эксперимента худший результат оказался у SelectKBest (R² = 0.7931), так как этот метод оценивает влияние признаков по отдельности и не учитывает их взаимодействия.

Методы RFE и RandomForest Importance показали себя лучше. RFE дал результат R² = 0.8044, а лучший среди сокращённых моделей — RandomForest Importance (R² = 0.8057). Это объясняется тем, что случайный лес лучше улавливает нелинейные зависимости в данных и более точно определяет важность признаков.

В итоге можно сказать, что уменьшение числа признаков приводит к небольшому снижению качества модели, но среди методов отбора признаков наилучшим оказался RandomForest Importance. Он обеспечил наименьшее падение точности по сравнению с базовой моделью.